<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2024/2025  

<img src="../../img/portada.png" width="300"/>

---

#  Ensamble Secuencial de Modelos Predictivos  
### Primera Convocatoria – Junio 2025  

**Profesor:** Juan Galán Páez  
**Autores:**  
Carlos Martín de Prado Barragán  
Marco Padilla Gómez  

**Fecha de entrega:** 9 de Junio de 2025  

</div>

## Introducción

En este notebook se realiza una exploración exhaustiva de los hiperparámetros del meta-modelo secuencial aplicado al conjunto de datos sobre la enfermedad de Parkinson.

Mediante validación cruzada con 5 particiones, se evalúan diferentes combinaciones de parámetros clave como el número de estimadores, la tasa de aprendizaje y el numero de ejemplos.

Además del coeficiente de determinación R², se incluyen métricas complementarias como el Error Absoluto Medio (MAE) y la Raíz del Error Cuadrático Medio (RMSE) para proporcionar una evaluación más completa del rendimiento.

El objetivo es identificar las configuraciones que maximicen la capacidad predictiva y la robustez del modelo, fundamentando así las decisiones de diseño y optimización.

## 1. Importación de librerías y configuración del entorno

- Se añade la ruta `../../src` para importar el módulo personalizado `sequential_ensemble.py` que contiene la implementación del meta-modelo secuencial.
- Se importan las librerías necesarias para el procesamiento de datos (`numpy`, `pandas`), modelado (`LinearRegression`), preprocesamiento (`StandardScaler`), validación cruzada (`cross_validate`, `KFold`), y métricas de evaluación (`mean_absolute_error`, `mean_squared_error`).
- Finalmente, se importa la clase `SequentialEnsembleRegressor` para utilizar el meta-modelo en los experimentos.

In [1]:
# Importación de librerías y configuración del entorno
import sys
sys.path.append('../../src')

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sequential_ensemble import SequentialEnsembleRegressor

## 2. Carga y preprocesamiento de datos

- Se carga el conjunto de datos `parkinsons.csv` desde la carpeta `data`.
- Se separan las variables predictoras (`X`) de la variable objetivo (`y`), que en este caso es la columna `total_UPDRS`.
- Se aplica estandarización a las variables predictoras utilizando `StandardScaler`, para normalizar su escala y facilitar el entrenamiento del modelo.


In [2]:
# Carga y preprocesamiento de datos
df = pd.read_csv("../../data/parkinsons.csv")
X = df.drop(columns=["total_UPDRS"])
y = df["total_UPDRS"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Definición del espacio de búsqueda de hiperparámetros

- Se establecen los rangos de valores para los hiperparámetros clave a explorar:
  - `n_estimators`: número de modelos base en el ensamblador, con valores 30, 50 y 200.
  - `learning_rate`: tasa de aprendizaje que modula la contribución de cada modelo, con valores 0.05, 0.2 y 0.5.
  - `sample_size`: porcentaje del conjunto de entrenamiento en cada iteración.
- Se inicializa una lista vacía `results` para almacenar los resultados de cada combinación probada durante la experimentación.

In [3]:
# Definimos rangos para los hiperparámetros más relevantes
n_estimators_options = [30, 50, 200]
learning_rate_options = [0.05, 0.2, 0.5]
sample_size_options = [0.75, 0.9, 1]  

# Para guardar resultados
results = []

## 4. Ejecución de experimentos con validación cruzada

- Se configura la validación cruzada con 5 particiones (`KFold`), mezclando aleatoriamente los datos para mayor representatividad.
- Se realiza un bucle anidado para probar todas las combinaciones posibles de los hiperparámetros definidos: número de estimadores, tasa de aprendizaje y porcentaje del conjunto de entreamiento.
- Para cada configuración, se crea una instancia del meta-modelo secuencial con esos parámetros.
- Se ejecuta la validación cruzada calculando múltiples métricas:  
  - Coeficiente de determinación R²,  
  - Error Absoluto Medio (MAE),  
  - Raíz del Error Cuadrático Medio (RMSE).
- Los resultados (medias y desviaciones estándar) se almacenan en la lista `results` para su posterior análisis.
- Se imprime un resumen en cada iteración para seguimiento en tiempo real de los experimentos.

In [6]:
# Configurar validación cruzada
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Métricas a evaluar como diccionario para evitar NaN
scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}

# Bucle para probar combinaciones con validación cruzada
for sample_size in sample_size_options:
    for n_estimators in n_estimators_options:
        for lr in learning_rate_options:
            modelo = SequentialEnsembleRegressor(
                base_estimator=LinearRegression,
                n_estimators=n_estimators,
                lr=lr,
                sample_size=sample_size,
                est_params={},
                random_state=42
            )
            scores = cross_validate(modelo, X_scaled, y, cv=kf, scoring=scoring)

            mean_r2 = np.mean(scores['test_r2'])
            mean_mae = -np.mean(scores['test_mae'])
            mean_rmse = np.sqrt(-np.mean(scores['test_mse']))
            std_r2 = np.std(scores['test_r2'])

            results.append({
                "sample_size": sample_size,
                "n_estimators": n_estimators,
                "learning_rate": lr,
                "mean_r2": mean_r2,
                "std_r2": std_r2,
                "mean_mae": mean_mae,
                "mean_rmse": mean_rmse
            })

            print(f"Evaluado sample_size={sample_size}, n_estimators={n_estimators}, "
                    f"lr={lr} -> R2 mean: {mean_r2:.4f}")

Evaluado sample_size=0.75, n_estimators=30, lr=0.05 -> R2 mean: -0.1726
Evaluado sample_size=0.75, n_estimators=30, lr=0.2 -> R2 mean: 0.1688
Evaluado sample_size=0.75, n_estimators=30, lr=0.5 -> R2 mean: 0.1686
Evaluado sample_size=0.75, n_estimators=50, lr=0.05 -> R2 mean: 0.1230
Evaluado sample_size=0.75, n_estimators=50, lr=0.2 -> R2 mean: 0.1633
Evaluado sample_size=0.75, n_estimators=50, lr=0.5 -> R2 mean: 0.1610
Evaluado sample_size=0.75, n_estimators=200, lr=0.05 -> R2 mean: 0.1651
Evaluado sample_size=0.75, n_estimators=200, lr=0.2 -> R2 mean: 0.1654
Evaluado sample_size=0.75, n_estimators=200, lr=0.5 -> R2 mean: 0.1668
Evaluado sample_size=0.9, n_estimators=30, lr=0.05 -> R2 mean: -0.1728
Evaluado sample_size=0.9, n_estimators=30, lr=0.2 -> R2 mean: 0.1678
Evaluado sample_size=0.9, n_estimators=30, lr=0.5 -> R2 mean: 0.1679
Evaluado sample_size=0.9, n_estimators=50, lr=0.05 -> R2 mean: 0.1239
Evaluado sample_size=0.9, n_estimators=50, lr=0.2 -> R2 mean: 0.1658
Evaluado sample

## 5. Análisis y visualización de resultados

- Se convierte la lista de resultados en un DataFrame de Pandas para facilitar su manipulación.
- Se ordenan las configuraciones probadas según el valor medio del coeficiente de determinación R², de mayor a menor, para identificar rápidamente las mejores combinaciones.
- Se reinicia el índice del DataFrame para mantener la consistencia tras el ordenamiento.
- Finalmente, se muestra la tabla con todas las configuraciones evaluadas y sus respectivos resultados, facilitando el análisis comparativo.

In [7]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="mean_r2", ascending=False)
results_df.reset_index(drop=True, inplace=True)
results_df

,sample_size,n_estimators,learning_rate,mean_r2,std_r2,mean_mae,mean_rmse
0,0.75,30,0.20,0.168820,0.034065,8.039355,9.680919
1,0.75,30,0.50,0.168555,0.032841,8.045530,9.682359
2,1.00,30,0.20,0.168069,0.034491,8.045663,9.685615
3,1.00,200,0.05,0.168049,0.034523,8.050643,9.685719
4,1.00,50,0.20,0.168048,0.034524,8.050729,9.685721
5,1.00,30,0.50,0.168048,0.034524,8.050789,9.685723
6,1.00,50,0.50,0.168048,0.034524,8.050789,9.685723
7,1.00,200,0.20,0.168048,0.034524,8.050789,9.685723
8,1.00,200,0.50,0.168048,0.034524,8.050789,9.685723
9,0.90,30,0.50,0.167920,0.035883,8.045006,9.686920


In [10]:
# Comparación del mejor ensamblador con el modelo base individual (DecisionTreeRegressor)
from sklearn.model_selection import cross_val_score

# Extraer la mejor configuración del ensamblador
mejor_config = results_df.iloc[0]

# Crear y evaluar el modelo base individual
modelo_base = LinearRegression()
r2_base = np.mean(cross_val_score(modelo_base, X_scaled, y, cv=kf, scoring='r2'))

# Mostrar comparación
print(f"R² del mejor ensamblador (validación cruzada): {mejor_config['mean_r2']:.4f}")
print(f"R² del modelo base individual: {r2_base:.4f}")

R² del mejor ensamblador (validación cruzada): 0.1688
R² del modelo base individual: 0.1680


## 6. Interpretación de resultados y conclusiones

Los experimentos realizados evidencian que el rendimiento del meta-modelo secuencial entrenado con `LinearRegression` está condicionado por la configuración de sus hiperparámetros, aunque las diferencias en desempeño son sutiles debido a la naturaleza lineal del estimador base.

- Cuando se emplea una tasa de aprendizaje baja (`lr=0.05`), el modelo tiene dificultades para ajustarse adecuadamente a los datos, obteniendo incluso valores negativos de R² en ciertas configuraciones con menor número de estimadores. Esto indica que una tasa de corrección tan reducida puede ser insuficiente para mejorar iterativamente las predicciones.

- Al incrementar la tasa de aprendizaje (`lr`) y el número de estimadores (`n_estimators`), el rendimiento mejora ligeramente, aunque no de forma sustancial. El mejor rendimiento se alcanzó con `sample_size=0.75`, `n_estimators=30` y `lr=0.2`, logrando un R² promedio de **0.1688**, apenas superior al obtenido por el modelo base individual `LinearRegression`, que fue de **0.1680**.

- El hiperparámetro `sample_size`, que define la proporción de ejemplos utilizados en cada iteración, no tuvo un impacto significativo en este caso, aunque valores alrededor de 0.75 o 0.9 permitieron mantener una ligera consistencia entre las pruebas.

- La baja variabilidad en los resultados sugiere que, con un estimador lineal, el meta-modelo no introduce suficiente flexibilidad como para superar de manera consistente al modelo individual, ya que todos los submodelos aprenden con la misma capacidad lineal limitada.

- A pesar de ello, el meta-modelo demuestra **comportamiento estable** y su arquitectura permite una **extensión sencilla a estimadores más complejos**, que podrían beneficiarse más del esquema secuencial con corrección de errores.

- En conclusión, aunque la mejora cuantitativa ha sido mínima en este caso, el enfoque de ensamble secuencial **no penaliza el rendimiento y es capaz de igualar o superar levemente a un modelo base simple**, lo que lo convierte en una técnica potencialmente valiosa si se combina con algoritmos no lineales más expresivos.
